In [ ]:
!pip install numpy pandas torch gymnasium transformers matplotlib

In [ ]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium

In [ ]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import mplfinance as mpf
import pandas_ta as ta
import pygame
import os
import pytz
import json
import re

from IPython.display import display, clear_output

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.callbacks import Callback, ModelCheckpoint
from scipy.signal import argrelextrema
import tensorflow as tf

In [ ]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 25  # 25 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 15  # 15 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [ ]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

index_symbol, quantity = get_index_symbol_and_quantity("Bank Nifty")

interval_minutes = 2 # Set the interval to 1, 5, or 15 minutes

ist_timezone = pytz.timezone("Asia/Kolkata")

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [ ]:
session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

In [ ]:
def fetch_candle_data(number):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                return result
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [ ]:
def fetch_train_candle_data(days_count):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [ ]:
def find_local_extrema(df):
    order=5
    atr_multiplier=1.5
    min_distance=5

    # Find local maxima and minima
    local_max = argrelextrema(df['high'].values, np.greater_equal, order=order)[0]
    local_min = argrelextrema(df['low'].values, np.less_equal, order=order)[0]

    # Calculate the threshold based on ATR
    threshold = df['ATR_14'] * atr_multiplier

    # Filter by significance
    significant_max = []
    significant_min = []

    for idx in local_max:
        if idx > order and idx < len(df) - order:
            high = df['high'].iloc[idx]
            if significant_min:
                low = df['low'].iloc[significant_min[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_max.append(idx)
            else:
                significant_max.append(idx)

    for idx in local_min:
        if idx > order and idx < len(df) - order:
            low = df['low'].iloc[idx]
            if significant_max:
                high = df['high'].iloc[significant_max[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_min.append(idx)
            else:
                significant_min.append(idx)

    # Ensure minimum distance
    def filter_by_distance(points, min_distance):
        filtered_points = []
        for i in range(len(points)):
            if not filtered_points or (points[i] - filtered_points[-1]) > min_distance:
                filtered_points.append(points[i])
        return filtered_points

    significant_max = filter_by_distance(significant_max, min_distance)
    significant_min = filter_by_distance(significant_min, min_distance)

    return significant_max, significant_min

In [ ]:
train_candles = fetch_candle_data(100)

train_df = pd.DataFrame(train_candles['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])

#train_df = fetch_train_candle_data(25)

print(len(train_df))

train_df = train_df.drop_duplicates(subset='datetime', keep='first')

print(len(train_df))

In [ ]:
# Ensure GPU/TPU availability
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class MarketAnalysisAgent:
    """
    Advanced Market Analysis Agent for analyzing market data (OHLC) and generating insights.
    """
    def __init__(self, data):
        self.data = data

    def preprocess_data(self):
        """Preprocess the market data, including datetime conversion and feature engineering."""
        ist = timezone('Asia/Kolkata')

        # Convert datetime to local timezone
        self.data['datetime'] = pd.to_datetime(self.data['datetime'], unit='s')
        self.data['datetime'] = (
            self.data['datetime']
            .dt.tz_localize('UTC')
            .dt.tz_convert(ist)
            .dt.tz_localize(None)
        )

        # Validate for duplicates or missing values in 'datetime'
        if self.data['datetime'].duplicated().any() or self.data['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Set datetime as index and drop unnecessary columns
        self.data.set_index(self.data['datetime'], inplace=True)
        self.data.drop(['datetime', 'volume'], axis=1, inplace=True, errors='ignore')

        # Add time-based features
        self.data['hour_of_day'] = self.data.index.hour
        self.data['day_of_week'] = self.data.index.dayofweek

        # Add price range features
        self.data['high_low_range'] = self.data['high'] - self.data['low']
        self.data['open_close_range'] = self.data['open'] - self.data['close']

        # Add technical indicators
        lengths = [14]

        for length in lengths:
            self.data[f'EMA_{length}'] = ta.ema(self.data['close'], length=length)
            self.data[f'RSI_{length}'] = ta.rsi(self.data['close'], length=length)
            self.data[f'ATR_{length}'] = ta.atr(self.data['high'], self.data['low'], self.data['close'], length=length)

        # Define targets and stop-loss based on ATR
        self.data['Target'] = self.data['ATR_14'] * 2
        self.data['Stop Loss'] = self.data['ATR_14']

        # Round values and drop NaNs
        self.data = self.data.round(2)
        self.data.dropna(inplace=True)

        return self.data

    def analyze(self):
        """Run the full analysis pipeline."""
        self.preprocess_data()
        self.label_signals()
        self.data = self.data[[col for col in self.data.columns if col not in ['Entry Price', 'Exit Price']]]
        return self.data

# Instantiate and analyze the data
agent = MarketAnalysisAgent(train_df)
analyzed_data = agent.analyze()
analyzed_data.tail()


In [ ]:
class SignalGenerationAgent:
    """
    Generates buy/sell signals based on analyzed data and predefined strategies.
    """
    def __init__(self, data):
        self.data = data

        def label_signals(self):
        """Label buy/sell signals based on target and stop-loss levels."""
        self.data['Signal'] = 0
        self.data['Entry Price'] = 0.0
        self.data['Exit Price'] = 0.0

        for index, row in self.data.iterrows():
            entry_price = row['close']
            target = row['Target']
            stop_loss = row['Stop Loss']

            # Define price thresholds
            buy_target_price = entry_price + target
            buy_sl_price = entry_price - stop_loss
            sell_target_price = entry_price - target
            sell_sl_price = entry_price + stop_loss

            future_data = self.data.loc[index:].iloc[1:]

            # Check for buy signal
            for future_index, future_row in future_data.iterrows():
                if future_row['high'] >= buy_target_price:
                    self.data.at[index, 'Signal'] = 2  # Buy Signal
                    self.data.at[index, 'Entry Price'] = entry_price
                    self.data.at[index, 'Exit Price'] = future_row['high']
                    break
                elif future_row['low'] <= buy_sl_price:
                    break

            # Check for sell signal
            for future_index, future_row in future_data.iterrows():
                if future_row['low'] <= sell_target_price:
                    self.data.at[index, 'Signal'] = 1  # Sell Signal
                    self.data.at[index, 'Entry Price'] = entry_price
                    self.data.at[index, 'Exit Price'] = future_row['low']
                    break
                elif future_row['high'] >= sell_sl_price:
                    break

        return self.data

   def analyze(self):
        """Run the full signal pipeline."""
        self.label_signals()
        self.data = self.data[[col for col in self.data.columns if col not in ['Entry Price', 'Exit Price']]]
        return self.data

# Generate signals
signal_generation_agent = SignalGenerationAgent(analyzed_data)
signal_data = signal_generation_agent.analyze()

signal_data.tail()

In [ ]:
# Portfolio Optimization Agent
class PortfolioOptimizationAgent:
    """
    Optimizes the portfolio allocation based on risk, returns, and constraints.
    """
    def optimize(self):
        pass  # Placeholder for future implementation

# Risk Management Agent
class RiskManagementAgent:
    """
    Manages risk by setting stop-loss, target levels, and position sizing.
    """
    def manage_risk(self):
        pass  # Placeholder for future implementation

# Execution Agent
class ExecutionAgent:
    """
    Executes trades efficiently in the live market.
    """
    def execute_trade(self):
        pass  # Placeholder for future implementation

# Adaptation Agent
class AdaptationAgent:
    """
    Adapts the model to changing market conditions using Meta-RL and online learning.
    """
    def adapt(self):
        pass  # Placeholder for future implementation

In [ ]:
import gymnasium as gym
from torch import nn, optim
from collections import deque
import random

# Define the environment for trading
class TradingEnv(gym.Env):
    def __init__(self, data):
        self.data = data
        self.current_step = 0
        self.done = False
        self.memory = deque(maxlen=10000)  # Store past decisions/strategies
        self.n_features = data.shape[1]  # Dynamically detect the number of features (columns)

    def step(self, action):
        if self.current_step >= len(self.data) - 1:  # Check if we are at the last row
            self.done = True
        else:
            self.current_step += 1

        next_state = self._next_observation()
        reward = 0  # Placeholder for reward logic
        return next_state, reward, self.done, {}

    def _next_observation(self):
        # Convert the row to a tensor and ensure the dimensions match
        state = self.data.iloc[self.current_step].values
        return state

    def reset(self):
        self.current_step = 0
        self.done = False
        return self._next_observation()


# Define a simple Meta-RL agent
class MetaRLAgent(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MetaRLAgent, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
# Train the agent
# Dynamically set input dimension based on the number of features
input_dim = signal_data.shape[1]  # Number of features (columns)
output_dim = 3  # Number of possible actions (e.g., Buy, Sell, Hold)

# Instantiate the agent
agent = MetaRLAgent(input_dim=input_dim, output_dim=output_dim).to(device)

# Define the training loop
def train_meta_rl(env, agent, episodes=1000):
    optimizer = optim.Adam(agent.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    for episode in range(episodes):
        state = env.reset()
        total_reward = 0
        while not env.done:
            state_tensor = torch.tensor(np.array(state), dtype=torch.float32).to(device)
            action = agent(state_tensor).argmax().item()  # Select action based on highest output
            next_state, reward, done, _ = env.step(action)

            # Store in memory
            env.memory.append((state, action, reward, next_state, done))
            total_reward += reward

            # Train the model using experiences from the memory
            if len(env.memory) > 64:
                batch = random.sample(env.memory, 64)
                states, actions, rewards, next_states, dones = zip(*batch)
                # Convert batch to tensors
                states_tensor = torch.tensor(np.array(states), dtype=torch.float32).to(device)

                actions_tensor = torch.tensor(actions, dtype=torch.long).to(device)
                rewards_tensor = torch.tensor(rewards, dtype=torch.float32).to(device)
                next_states_tensor = torch.tensor(np.array(next_states), dtype=torch.float32).to(device)
                dones_tensor = torch.tensor(dones, dtype=torch.bool).to(device)

                # Forward pass
                q_values = agent(states_tensor)
                next_q_values = agent(next_states_tensor)

                # Calculate target Q-values
                target_q_values = rewards_tensor + (0.99 * next_q_values.max(dim=1)[0] * (1 - dones_tensor.float()))

                # Calculate loss and optimize
                loss = criterion(q_values.gather(1, actions_tensor.unsqueeze(1)), target_q_values.unsqueeze(1))
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            state = next_state
        print(f"Episode {episode + 1}, Total Reward: {total_reward}")

# Instantiate and train the model
env = TradingEnv(signal_data)
train_meta_rl(env, agent)

In [ ]:
def sanitize_filename(filename):
    # Define a regex pattern for allowed characters (alphanumeric and some special characters)
    pattern = re.compile(r'[^a-zA-Z0-9_.-]')
    # Replace any character not matching the pattern with an underscore
    sanitized_filename = pattern.sub('_', filename)
    return sanitized_filename

In [ ]:
# Function to check if the file is from the past week
def is_file_from_past_week(filepath):
    if not os.path.exists(filepath):
        return False
    file_mod_time = datetime.fromtimestamp(os.path.getmtime(filepath))
    return file_mod_time.date() >= (datetime.today().date() - timedelta(days=7))

In [ ]:
class PlotAccuracy(Callback):
    def on_train_begin(self, logs={}):
        self.i = 0
        self.x = []
        self.accuracies = []
        self.val_accuracies = []

        # Apply dark background style to Matplotlib
        plt.style.use('dark_background')

        self.fig = plt.figure()

        self.logs = []

    def on_epoch_end(self, epoch, logs={}):
        self.logs.append(logs)
        self.x.append(self.i)
        self.accuracies.append(logs.get('accuracy'))
        self.val_accuracies.append(logs.get('val_accuracy'))
        self.i += 1

        # Clear previous plot and plot new data
        clear_output(wait=True)

        plt.figure(figsize=(14, 7))

        plt.plot(self.x, self.accuracies, label="Training Accuracy", color='white')
        plt.plot(self.x, self.val_accuracies, label="Validation Accuracy", color=(0.95, 0.38, 0.25, 1))
        plt.legend()
        plt.title('Training Accuracy vs. Validation Accuracy', color='white')
        plt.xlabel('Epoch', color='white')
        plt.ylabel('Accuracy', color='white')

        # Customize tick colors
        plt.tick_params(colors='white')

        plt.grid(False)

        plt.show()

In [ ]:
def preprocess_trend_classification_data(df, sequence_length):
    scalers = {}
    for column in df.columns:
        scalers[column] = MinMaxScaler()
        df[column] = scalers[column].fit_transform(df[column].values.reshape(-1, 1))

    label_encoder_trend = LabelEncoder()
    df['Trend'] = label_encoder_trend.fit_transform(df['Trend'])

    X_trend_clf, y_trend_clf = [], []
    for i in range(sequence_length, len(df)):
        X_trend_clf.append(df.iloc[i-sequence_length:i].values)
        y_trend_clf.append(df.iloc[i, df.columns.get_loc('Trend')])

    X_trend_clf, y_trend_clf = np.array(X_trend_clf), np.array(y_trend_clf)
    return X_trend_clf, y_trend_clf, scalers, label_encoder_trend

def preprocess_signal_classification_data(df, sequence_length):
    scalers = {}
    for column in df.columns:
        scalers[column] = MinMaxScaler()
        df[column] = scalers[column].fit_transform(df[column].values.reshape(-1, 1))

    label_encoder_signal = LabelEncoder()
    df['Signal'] = label_encoder_signal.fit_transform(df['Signal'])

    X_signal_clf, y_signal_clf = [], []
    for i in range(sequence_length, len(df)):
        X_signal_clf.append(df.iloc[i-sequence_length:i].values)
        y_signal_clf.append(df.iloc[i, df.columns.get_loc('Signal')])

    X_signal_clf, y_signal_clf = np.array(X_signal_clf), np.array(y_signal_clf)
    return X_signal_clf, y_signal_clf, scalers, label_encoder_signal

# Set sequence length
sequence_length = 5

# Trend Classification Data
X_trend_clf, y_trend_clf, trend_scalers, trend_label_encoder = preprocess_trend_classification_data(
    train_df.copy(), sequence_length
)
test_size_trend_clf = int(len(X_trend_clf) * 0.2)
X_train_trend_clf, X_test_trend_clf = X_trend_clf[:-test_size_trend_clf], X_trend_clf[-test_size_trend_clf:]
y_train_trend_clf, y_test_trend_clf = y_trend_clf[:-test_size_trend_clf], y_trend_clf[-test_size_trend_clf:]

# Signal Classification Data
X_signal_clf, y_signal_clf, signal_scalers, signal_label_encoder = preprocess_signal_classification_data(
    train_df.copy(), sequence_length
)
test_size_signal_clf = int(len(X_signal_clf) * 0.2)
X_train_signal_clf, X_test_signal_clf = X_signal_clf[:-test_size_signal_clf], X_signal_clf[-test_size_signal_clf:]
y_train_signal_clf, y_test_signal_clf = y_signal_clf[:-test_size_signal_clf], y_signal_clf[-test_size_signal_clf:]

In [ ]:
def build_bidirectional_lstm_attention_model_trend_clf(input_shape, num_trend_classes_clf):
    inputs = Input(shape=input_shape)

    # First LSTM Layer
    lstm = LSTM(512, return_sequences=True)(inputs)
    attention = Attention()([lstm, lstm])
    x = Dropout(0.3)(attention)

    # Second LSTM Layer
    x = LSTM(256, return_sequences=True)(x)
    x = Dropout(0.3)(x)

    # Third LSTM Layer
    x = LSTM(128, return_sequences=False)(x)
    x = Dropout(0.3)(x)

    # Classification output layer
    outputs = Dense(num_trend_classes_clf, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model

# Trend Classification Variables
trend_lstm_clf_filename = f'lstm_trend_clf_model_{index_symbol}_{interval_minutes}.keras'
trend_lstm_clf_path = f'models/{sanitize_filename(trend_lstm_clf_filename)}'
num_trend_classes_clf = len(np.unique(y_train_trend_clf))

# Load or Train Trend Model
if os.path.exists(trend_lstm_clf_path) and is_file_from_past_week(trend_lstm_clf_path):
    trend_lstm_clf_model = load_model(trend_lstm_clf_path)
    print("Loaded existing LSTM trend classification model.")
else:
    input_shape_trend_clf = (X_train_trend_clf.shape[1], X_train_trend_clf.shape[2])
    trend_lstm_clf_model = build_bidirectional_lstm_attention_model_trend_clf(input_shape_trend_clf, num_trend_classes_clf)
    trend_lstm_clf_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stopping_trend_clf = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
    reduce_lr_trend_clf = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, min_lr=0.00001)
    checkpoint_trend_clf = ModelCheckpoint(trend_lstm_clf_path, monitor='val_accuracy', save_best_only=True, save_weights_only=False, mode='max')

    trend_lstm_clf_model.fit(
        X_train_trend_clf, y_train_trend_clf,
        validation_data=(X_test_trend_clf, y_test_trend_clf),
        epochs=100,
        batch_size=64,
        callbacks=[early_stopping_trend_clf, reduce_lr_trend_clf, checkpoint_trend_clf, PlotAccuracy()],
        verbose=1
    )
    print("Trained and saved a new LSTM trend classification model.")

# Evaluate Trend Classification Model
y_pred_trend_lstm_clf = trend_lstm_clf_model.predict(X_test_trend_clf)
y_pred_trend_lstm_clf = np.argmax(y_pred_trend_lstm_clf, axis=1)
accuracy_trend_lstm_clf = accuracy_score(y_test_trend_clf, y_pred_trend_lstm_clf)
print(f'LSTM Trend Classification Model Accuracy: {accuracy_trend_lstm_clf}')
print(classification_report(y_test_trend_clf, y_pred_trend_lstm_clf))

In [ ]:
def build_bidirectional_lstm_attention_model_signal_clf(input_shape, num_signal_classes_clf):
    inputs = Input(shape=input_shape)

    # First LSTM Layer
    lstm = LSTM(512, return_sequences=True)(inputs)
    attention = Attention()([lstm, lstm])
    x = Dropout(0.3)(attention)

    # Second LSTM Layer
    x = LSTM(256, return_sequences=True)(x)
    x = Dropout(0.3)(x)

    # Third LSTM Layer
    x = LSTM(128, return_sequences=False)(x)
    x = Dropout(0.3)(x)

    # Classification output layer
    outputs = Dense(num_signal_classes_clf, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model

# Signal Classification Variables
signal_lstm_clf_filename = f'lstm_signal_clf_model_{index_symbol}_{interval_minutes}.keras'
signal_lstm_clf_path = f'models/{sanitize_filename(signal_lstm_clf_filename)}'
num_signal_classes_clf = len(np.unique(y_train_signal_clf))

# Load or Train Signal Model
if os.path.exists(signal_lstm_clf_path) and is_file_from_past_week(signal_lstm_clf_path):
    signal_lstm_clf_model = load_model(signal_lstm_clf_path)
    print("Loaded existing LSTM signal classification model.")
else:
    input_shape_signal_clf = (X_train_signal_clf.shape[1], X_train_signal_clf.shape[2])
    signal_lstm_clf_model = build_bidirectional_lstm_attention_model_signal_clf(input_shape_signal_clf, num_signal_classes_clf)
    signal_lstm_clf_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stopping_signal_clf = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
    reduce_lr_signal_clf = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, min_lr=0.00001)
    checkpoint_signal_clf = ModelCheckpoint(signal_lstm_clf_path, monitor='val_accuracy', save_best_only=True, save_weights_only=False, mode='max')

    signal_lstm_clf_model.fit(
        X_train_signal_clf, y_train_signal_clf,
        validation_data=(X_test_signal_clf, y_test_signal_clf),
        epochs=100,
        batch_size=64,
        callbacks=[early_stopping_signal_clf, reduce_lr_signal_clf, checkpoint_signal_clf, PlotAccuracy()],
        verbose=1
    )
    print("Trained and saved a new LSTM signal classification model.")

# Evaluate Signal Classification Model
y_pred_signal_lstm_clf = signal_lstm_clf_model.predict(X_test_signal_clf)
y_pred_signal_lstm_clf = np.argmax(y_pred_signal_lstm_clf, axis=1)
accuracy_signal_lstm_clf = accuracy_score(y_test_signal_clf, y_pred_signal_lstm_clf)
print(f'LSTM Signal Classification Model Accuracy: {accuracy_signal_lstm_clf}')
print(classification_report(y_test_signal_clf, y_pred_signal_lstm_clf))

In [ ]:
def format_capital(capital):
    # Check if the value is negative
    negative = capital < 0
    # Use the absolute value for formatting
    capital = abs(capital)

    if capital >= 1_00_00_00_00_00_00_000:  # 1 Shankh
        formatted_value = f'{capital / 1_00_00_00_00_00_00_000:.2f} Shankh'
    elif capital >= 1_00_00_00_00_00_00_000:  # 1 Padma
        formatted_value = f'{capital / 1_00_00_00_00_00_00_000:.2f} Padma'
    elif capital >= 1_00_00_00_00_00_000:  # 1 Nil
        formatted_value = f'{capital / 1_00_00_00_00_00_000:.2f} Nil'
    elif capital >= 1_00_00_00_00_000:  # 1 Kharab
        formatted_value = f'{capital / 1_00_00_00_00_000:.2f} Kharab'
    elif capital >= 1_00_00_00_000:  # 1 Arab
        formatted_value = f'{capital / 1_00_00_00_000:.2f} Arab'
    elif capital >= 1_00_00_000:  # 1 Crore
        formatted_value = f'{capital / 1_00_00_000:.2f} Cr'
    elif capital >= 1_00_000:  # 1 Lakh
        formatted_value = f'{capital / 1_00_000:.2f} L'
    elif capital >= 1_000:  # 1 Thousand
        formatted_value = f'{capital / 1_000:.2f} K'
    else:
        formatted_value = f'{capital:.2f}'

    # Prepend the negative sign if needed
    return f'-{formatted_value}' if negative else formatted_value

In [ ]:
# Function to format trade time
def format_trade_time(seconds):
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)

    if hours > 0:
        return f"{int(hours)} Hour{'s' if hours > 1 else ''} {int(minutes)} Min{'s' if minutes > 1 else ''}"
    elif minutes > 0:
        return f"{int(minutes)} Min{'s' if minutes > 1 else ''}"
    else:
        return f"{int(seconds)} Sec{'S' if seconds > 1 else ''}"

In [ ]:
backtest_candle_data = fetch_candle_data(20)

raw_backtest_df = pd.DataFrame(backtest_candle_data['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])

print(len(raw_backtest_df))

raw_backtest_df = raw_backtest_df.drop_duplicates(subset='datetime', keep='first')
raw_backtest_df

print(len(raw_backtest_df))

In [ ]:
initial_capital = 10000
backtest_results = []

crop_fetched_candle_data = 599

def backtest_logic():
    global backtest_results

    backtest_capital = initial_capital
    backtest_quantity = quantity
    temp_capital = initial_capital

    total_backtest_profits = 0
    total_backtest_losses = 0
    total_backtest_brokerage = 0

    backtest_trade_active = False
    backtest_entry_price = 0
    backtest_target_price = 0
    backtest_stop_loss_price = 0

    entry_type = None
    backtest_entry_datetime = None
    entry_trend_prediction = None
    entry_signal_prediction = None

    for i in range(crop_fetched_candle_data, len(raw_backtest_df)):
        if backtest_capital > 2 * temp_capital:
            backtest_quantity *= 2
            temp_capital *= 2

        backtest_df = raw_backtest_df[i-crop_fetched_candle_data:i+1].copy()

        backtest_df = process_df_with_features(backtest_df)
        backtest_df = label_signals(backtest_df)

        backtest_df = backtest_df[[col for col in backtest_df.columns if col not in ['Entry Price', 'Exit Price']]]

        print(len(backtest_df))

        backtest_df = backtest_df.iloc[:-1]

        current_time = backtest_df.index[-1].time()
        actual_closing_original = backtest_df['close'].iloc[-1]

        if not backtest_trade_active and current_time >= dt_time(9, (15 + interval_minutes)) and current_time <= dt_time(15, 0):
            # Trend Prediction using LSTM
            scaled_backtest_trend_df = backtest_df[-sequence_length:].copy()
            for column in scaled_backtest_trend_df.columns:
                scaled_backtest_trend_df[column] = trend_scalers[column].transform(scaled_backtest_trend_df[column].values.reshape(-1, 1))

            current_sequence_trend = scaled_backtest_trend_df.values
            current_sequence_trend = current_sequence_trend.reshape(1, sequence_length, -1)

            y_pred_trend_lstm_backtest = trend_lstm_clf_model.predict(current_sequence_trend)
            y_pred_trend_lstm_backtest = np.argmax(y_pred_trend_lstm_backtest, axis=1)

            # Signal Classification using LSTM
            scaled_backtest_signal_df = backtest_df[-sequence_length:].copy()
            for column in scaled_backtest_signal_df.columns:
                scaled_backtest_signal_df[column] = signal_scalers[column].transform(scaled_backtest_signal_df[column].values.reshape(-1, 1))

            current_sequence_signal = scaled_backtest_signal_df.values
            current_sequence_signal = current_sequence_signal.reshape(1, sequence_length, -1)

            y_pred_signal_lstm_backtest = signal_lstm_clf_model.predict(current_sequence_signal)
            y_pred_signal_lstm_backtest = np.argmax(y_pred_signal_lstm_backtest, axis=1)

            # Print Predictions
            print(f'Actual Closing Price: {actual_closing_original}')
            print(f'Trend Prediction (LSTM): {y_pred_trend_lstm_backtest}')
            print(f'Signal Prediction (LSTM): {y_pred_signal_lstm_backtest}')


        print(backtest_df.index[-1])

        if current_time >= dt_time(9, (15 + interval_minutes)) and current_time <= dt_time(15, 0):
            if not backtest_trade_active and entry_type == None:
                if y_pred_trend_lstm_backtest == 2:
                    backtest_entry_price = actual_closing_original

                    backtest_target_price = int(backtest_entry_price + backtest_df['Target'].iloc[-1])
                    backtest_stop_loss_price = int(backtest_entry_price - backtest_df['Stop Loss'].iloc[-1])

                    backtest_trade_active = True
                    entry_type = "CE"
                    backtest_entry_datetime = backtest_df.index[-1]

                    print(f"CE Signal at {backtest_df.index[-1]}")
                    print(f"Entry Price: {backtest_entry_price}")

                elif y_pred_trend_lstm_backtest == 1:
                    backtest_entry_price = actual_closing_original

                    backtest_target_price = int(backtest_entry_price - backtest_df['Target'].iloc[-1])
                    backtest_stop_loss_price = int(backtest_entry_price + backtest_df['Stop Loss'].iloc[-1])

                    backtest_trade_active = True
                    entry_type = "PE"
                    backtest_entry_datetime = backtest_df.index[-1]

                    print(f"PE Signal at {backtest_df.index[-1]}")
                    print(f"Entry Price: {backtest_entry_price}")

            else:
                if entry_type == "CE":
                    if actual_closing_original >= backtest_target_price:
                        points = int((backtest_target_price - backtest_entry_price))
                        profits = int(points * backtest_quantity)
                        backtest_capital += profits
                        total_backtest_profits += 1
                        total_backtest_brokerage += brokerage

                        win_percentage = round(total_backtest_profits / (total_backtest_profits + total_backtest_losses) * 100, 2)
                        loss_percentage = round(total_backtest_losses / (total_backtest_profits + total_backtest_losses) * 100, 2)

                        print(f"CE Target hit at {backtest_df.index[-1]}")
                        print(f"Profit: {format_capital(profits)}")
                        print(f'Profit/Loss %: {win_percentage}% / {loss_percentage}%')
                        print(f"Capital: {format_capital(backtest_capital)}")

                        backtest_results.append({
                            'Entry Time': backtest_entry_datetime,
                            'Exit Time': backtest_df.index[-1],
                            'Entry Type': entry_type,
                            'Entry Price': backtest_entry_price,
                            'Exit Price': actual_closing_original,
                            'Quantity': backtest_quantity,
                            'Points': points,
                            'Profit': format_capital(profits),
                            'Accuracy': f"{win_percentage}%",
                            'Capital': format_capital(backtest_capital),
                            'Brokerage': format_capital(total_backtest_brokerage),

                        })

                        backtest_trade_active = False
                        entry_type = None

                    elif actual_closing_original <= backtest_stop_loss_price:
                        points = int((backtest_stop_loss_price - backtest_entry_price))
                        profits = int(points * backtest_quantity)
                        backtest_capital += profits
                        total_backtest_losses += 1
                        total_backtest_brokerage += brokerage

                        win_percentage = round(total_backtest_profits / (total_backtest_profits + total_backtest_losses) * 100, 2)
                        loss_percentage = round(total_backtest_losses / (total_backtest_profits + total_backtest_losses) * 100, 2)

                        print(f"CE SL hit at {backtest_df.index[-1]}")
                        print(f"Loss: {format_capital(profits)}")
                        print(f'Profit/Loss %: {win_percentage}% / {loss_percentage}%')
                        print(f"Capital: {format_capital(backtest_capital)}")

                        backtest_results.append({
                            'Entry Time': backtest_entry_datetime,
                            'Exit Time': backtest_df.index[-1],
                            'Entry Type': entry_type,
                            'Entry Price': backtest_entry_price,
                            'Exit Price': actual_closing_original,
                            'Quantity': backtest_quantity,
                            'Points': points,
                            'Profit': format_capital(profits),
                            'Accuracy': f"{win_percentage}%",
                            'Capital': format_capital(backtest_capital),
                            'Brokerage': format_capital(total_backtest_brokerage),

                        })

                        backtest_trade_active = False
                        entry_type = None

                elif entry_type == "PE":
                    if actual_closing_original <= backtest_target_price:
                        points = int((backtest_entry_price - backtest_target_price))
                        profits = int(points * backtest_quantity)
                        backtest_capital += profits
                        total_backtest_profits += 1
                        total_backtest_brokerage += brokerage

                        win_percentage = round(total_backtest_profits / (total_backtest_profits + total_backtest_losses) * 100, 2)
                        loss_percentage = round(total_backtest_losses / (total_backtest_profits + total_backtest_losses) * 100, 2)

                        print(f"PE Target hit at {backtest_df.index[-1]}")
                        print(f"Profit: {format_capital(profits)}")
                        print(f'Profit/Loss %: {win_percentage}% / {loss_percentage}%')
                        print(f"Capital: {format_capital(backtest_capital)}")

                        backtest_results.append({
                            'Entry Time': backtest_entry_datetime,
                            'Exit Time': backtest_df.index[-1],
                            'Entry Type': entry_type,
                            'Entry Price': backtest_entry_price,
                            'Exit Price': actual_closing_original,
                            'Quantity': backtest_quantity,
                            'Points': points,
                            'Profit': format_capital(profits),
                            'Accuracy': f"{win_percentage}%",
                            'Capital': format_capital(backtest_capital),
                            'Brokerage': format_capital(total_backtest_brokerage),

                        })

                        backtest_trade_active = False
                        entry_type = None

                    elif actual_closing_original >= backtest_stop_loss_price:
                        points = int((backtest_entry_price - backtest_stop_loss_price))
                        profits = int(points * backtest_quantity)
                        backtest_capital += profits
                        total_backtest_losses += 1
                        total_backtest_brokerage += brokerage

                        win_percentage = round(total_backtest_profits / (total_backtest_profits + total_backtest_losses) * 100, 2)
                        loss_percentage = round(total_backtest_losses / (total_backtest_profits + total_backtest_losses) * 100, 2)

                        print(f"PE SL hit at {backtest_df.index[-1]}")
                        print(f"Loss: {format_capital(profits)}")
                        print(f'Profit/Loss %: {win_percentage}% / {loss_percentage}%')
                        print(f"Capital: {format_capital(backtest_capital)}")

                        backtest_results.append({
                            'Entry Time': backtest_entry_datetime,
                            'Exit Time': backtest_df.index[-1],
                            'Entry Type': entry_type,
                            'Entry Price': backtest_entry_price,
                            'Exit Price': actual_closing_original,
                            'Quantity': backtest_quantity,
                            'Points': points,
                            'Profit': format_capital(profits),
                            'Accuracy': f"{win_percentage}%",
                            'Capital': format_capital(backtest_capital),
                            'Brokerage': format_capital(total_backtest_brokerage),

                        })

                        backtest_trade_active = False
                        entry_type = None

        else:
            backtest_trade_active = False
            entry_type = None

        #clear_output(wait=True)

backtest_logic()

In [ ]:
backtest_results

In [ ]:
backtest_results_df = pd.DataFrame(backtest_results)

backtest_results_df.tail(15)

In [ ]:
def parse_capital(formatted_value):
    suffixes = {
        'Shankh': 1_00_00_00_00_00_00_000,
        'Padma': 1_00_00_00_00_00_00_000,
        'Nil': 1_00_00_00_00_00_000,
        'Kharab': 1_00_00_00_00_000,
        'Arab': 1_00_00_00_000,
        'Cr': 1_00_00_000,
        'L': 1_00_000,
        'K': 1_000
    }

    for suffix, multiplier in suffixes.items():
        if formatted_value.endswith(suffix):
            return float(formatted_value.replace(suffix, '').strip()) * multiplier

    return float(formatted_value)

# Convert the columns back to float
backtest_results_df['Inv_Capital'] = backtest_results_df['Capital'].apply(parse_capital)

In [ ]:
def currency(x, pos):
    return f'₹{format_capital(x)}'

# Calculate drawdown
backtest_results_df['Drawdown'] = backtest_results_df['Inv_Capital'].cummax() - backtest_results_df['Inv_Capital']
backtest_results_df['Drawdown Percent'] = (backtest_results_df['Drawdown'] / backtest_results_df['Inv_Capital'].cummax()) * 100

backtest_results_df['Temp_Profit'] = backtest_results_df['Inv_Capital'] - backtest_results_df['Drawdown']

# Plot drawdown
plt.style.use('dark_background')
plt.figure(figsize=(14, 14))

formatter = FuncFormatter(currency)

plt.subplot(2, 1, 1)
plt.plot(backtest_results_df.index, backtest_results_df['Inv_Capital'], color='green', label='Capital')
plt.ylabel('Capital', color='white')
plt.title('Capital Over Time', color='white')
plt.legend()
plt.grid(False)
plt.tick_params(colors='white')
plt.gca().yaxis.set_major_formatter(formatter)

plt.subplot(2, 1, 2)
plt.plot(backtest_results_df.index, -backtest_results_df['Drawdown'], label='Drawdown', color='red')
plt.ylabel('Drawdown', color='white')
plt.title('Drawdown Over Time', color='white')
plt.legend()
plt.grid(False)
plt.tick_params(colors='white')
plt.gca().yaxis.set_major_formatter(formatter)

plt.tight_layout()
plt.show()

In [ ]:
import gymnasium as gym
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from gymnasium import spaces
from collections import deque

class OptionTradingEnv(gym.Env):
    def __init__(self, df, initial_balance=10000, window_size=30, replay_buffer_size=1000):
        super(OptionTradingEnv, self).__init__()

        self.df = df
        self.initial_balance = initial_balance
        self.window_size = window_size
        self.n_features = df.shape[1]
        self.quantity = quantity
        self.replay_buffer_size = replay_buffer_size

        self.replay_buffer = deque(maxlen=self.replay_buffer_size)

        # Action space: 0 = Hold, 1 = Buy CE, 2 = Buy PE
        self.action_space = spaces.Discrete(3)

        # Observation space: Flattened window of past prices/features
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(window_size * self.n_features,), dtype=np.float32
        )

        # Track trade counts and profits/losses
        self.total_trades = 0
        self.total_ce_trades = 0
        self.total_pe_trades = 0
        self.total_profits = 0
        self.total_losses = 0
        self.total_points_captured = 0  # To track cumulative points captured or lost

    def reset(self, seed=None, **kwargs):
        super().reset(seed=seed)

        self.balance = self.initial_balance
        self.position = None  # None, 'CE', or 'PE'
        self.entry_price = None
        self.current_step = self.window_size
        self.done = False

        return self._get_observation(), {}

    def step(self, action):
        # Track initial balance to calculate the reward only when position is closed
        prev_balance = self.balance

        # Take action (buy CE, buy PE, or hold)
        if action == 1:  # Buy CE
            if self.position is None:
                self.position = "CE"
                self.entry_price = self.df.iloc[self.current_step]['close']
                self.target = abs(self.df.iloc[self.current_step]['Target'])
                self.stop_loss = abs(self.df.iloc[self.current_step]['Stop Loss'])
                self.total_ce_trades += 1
                self.total_trades += 1
        elif action == 2:  # Buy PE
            if self.position is None:
                self.position = "PE"
                self.entry_price = self.df.iloc[self.current_step]['close']
                self.target = abs(self.df.iloc[self.current_step]['Target'])
                self.stop_loss = abs(self.df.iloc[self.current_step]['Stop Loss'])
                self.total_pe_trades += 1
                self.total_trades += 1
        elif action == 0:  # Hold
            pass

        # Calculate reward only if a position is being closed
        reward = 0
        if self.position:
            current_price = self.df.iloc[self.current_step]['close']

            # For Call Option (CE)
            if self.position == "CE":
                # Stop Loss Check
                if (self.entry_price - current_price) >= self.stop_loss:
                    reward = -self.stop_loss / 100
                    self.balance += self.stop_loss * quantity
                    self.total_losses += 1
                    self.total_points_captured -= self.stop_loss
                    self.position = None  # Exit the position

                # Target Check
                elif (current_price - self.entry_price) >= self.target:
                    reward = self.target / 100
                    self.balance += self.target * quantity
                    self.total_profits += 1
                    self.total_points_captured += self.target
                    self.position = None  # Exit the position

            # For Put Option (PE)
            elif self.position == "PE":
                # Stop Loss Check
                if (current_price - self.entry_price) >= self.stop_loss:
                    reward = -self.stop_loss / 100
                    self.balance += self.stop_loss * quantity
                    self.total_losses += 1
                    self.total_points_captured -= self.stop_loss
                    self.position = None  # Exit the position

                # Target Check
                elif (self.entry_price - current_price) >= self.target:
                    reward = self.target / 100
                    self.balance += self.target * quantity
                    self.total_profits += 1
                    self.total_points_captured += self.target
                    self.position = None  # Exit the position

        # Increment step and determine if the episode should end
        self.current_step += 1
        if self.current_step >= len(self.df) - 1:
            self.done = True

        # Store experience in replay buffer
        experience = {
            "state": self._get_observation(),
            "action": action,
            "reward": reward,
            "next_state": self._get_observation(),
            "done": self.done
        }
        self.replay_buffer.append(experience)

        obs = self._get_observation()

        info = {
            "balance": self.balance,
            "total_trades": self.total_trades,
            "total_ce_trades": self.total_ce_trades,
            "total_pe_trades": self.total_pe_trades,
            "total_profits": self.total_profits,
            "total_losses": self.total_losses,
            "total_points_captured": self.total_points_captured,
        }

        return obs, reward, self.done, False, info

    def _get_observation(self):
        obs_window = self.df.iloc[self.current_step - self.window_size: self.current_step].values
        return obs_window.flatten().astype(np.float32)

# Initialize environment with 10,000 INR and a quantity (e.g., 1 option)
env = DummyVecEnv([lambda: OptionTradingEnv(train_df, initial_balance=10000)])

# Initialize PPO model
model = PPO('MlpPolicy', env, verbose=1)

# Train the model
model.learn(total_timesteps=100000)

# Save the model
model.save("ppo_option_trading_with_target_sl")

In [ ]:
# Testing the model

# Reload the trained model
model = PPO.load("ppo_option_trading_with_target_sl")

# Create the environment for testing (using the same train_df or a test set)
test_env = DummyVecEnv([lambda: OptionTradingEnv(train_df, initial_balance=10000)])

# Reset the environment to start a new episode
obs = test_env.reset()

done = False
total_reward = 0

while not done:
    # Predict the action using the trained model
    action, _states = model.predict(obs)

    # Take the action in the environment
    obs, reward, done, info = test_env.step(action)

    # Accumulate the reward for tracking the agent's performance
    total_reward += reward

# Access the final balance after the test run
final_balance = info[0]["balance"] # Correcting the list indexing

# Calculate profit or loss
profit_or_loss = final_balance - 10000 # Initial balance was 10,000 INR
print(f"Final Balance: {round(final_balance, 2)} INR")
print(f"Profit/Loss: {round(profit_or_loss, 2)} INR")

# Access trade stats
total_trades = info[0]["total_trades"]
total_ce_trades = info[0]["total_ce_trades"]
total_pe_trades = info[0]["total_pe_trades"]
total_profits = info[0]["total_profits"]
total_losses = info[0]["total_losses"]
total_points_captured = info[0]["total_points_captured"]
print(f"Total Trades: {total_trades}")
print(f"Total CE Trades: {total_ce_trades}")
print(f"Total PE Trades: {total_pe_trades}")
print(f"Total Profits: {total_profits}")
print(f"Total Losses: {total_losses}")
print(f"Total Points Captured: {int(total_points_captured)}")

In [ ]:
def get_sleep_time(interval_minutes, market_start_hour=9, market_start_minute=15):
    now = datetime.now()
    market_start_time = now.replace(hour=market_start_hour, minute=market_start_minute, second=0, microsecond=0)

    if now < market_start_time:
        # If current time is before the market starts, set next_run_time to market start time
        next_run_time = market_start_time
    else:
        # Calculate the minutes since the market start time
        minutes_since_market_start = (now - market_start_time).total_seconds() // 60
        # Calculate the number of minutes to the next interval boundary
        minutes_to_next_interval = interval_minutes - (minutes_since_market_start % interval_minutes)
        # Calculate the next run time by adding these minutes to the current time
        next_run_time = (now + timedelta(minutes=minutes_to_next_interval)).replace(second=0, microsecond=0)

    # Calculate the sleep time in seconds
    sleep_time = (next_run_time - now).total_seconds()
    return sleep_time

In [ ]:
def fetch_option_chain():
    while True:
        try:
            data = {
                "symbol": index_symbol,
                "strikecount": 2,
                "timestamp": ""
            }
            response = fyers.optionchain(data=data)

            if response is not None:
                return response
        except Exception as e:
            print(f"Error fetching Option Chain: {e}")
            time.sleep(active_order_sleep)

index_oc= fetch_option_chain()

pd.DataFrame(index_oc['data']['optionsChain'])

In [ ]:
def assign_ce_pe_option_symbols():
    symbol_oc = fetch_option_chain()

    if symbol_oc != None:
        # Convert the response data into a DataFrame
        oc_df = pd.DataFrame(symbol_oc['data']['optionsChain'])

        # Find the first 'CE' symbol from the top
        first_ce_symbol = None
        for index, row in oc_df.iterrows():
            if row['option_type'] == 'CE':
                first_ce_symbol = row['symbol']
                first_ce_strike = row['strike_price']
                break

        # Find the first 'PE' symbol from the bottom
        first_pe_symbol = None
        for index, row in oc_df[::-1].iterrows():  # Iterate in reverse
            if row['option_type'] == 'PE':
                first_pe_symbol = row['symbol']
                first_pe_strike = row['strike_price']
                break

        return first_ce_symbol, first_pe_symbol, first_ce_strike, first_pe_strike

In [ ]:
def onmessage_ce(ce_message):
    global ce_ltp, index_ltp, unsubscribe_done
    try:
        if ce_message['symbol'] == ce_symbol:
            if "ltp" in ce_message:
                ce_ltp = ce_message["ltp"]
                ce_ltp = float(ce_ltp)

        elif ce_message['symbol'] == index_symbol:
            if "ltp" in ce_message:
                index_ltp = ce_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [ce_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {ce_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessageCE): {e}")


def onerror_ce(message):
    print("CE Error:", message)


def onclose_ce(message):
    print("CE Connection closed:", message)


def onopen_ce():

    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [ce_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def ce_buy_sell_ltp():
    global buy_sell_checked, ce_symbol, ce_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching CE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if ce_symbol is not None and ce_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                ce_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_ce,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_ce,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_ce,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_ce             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                ce_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def onmessage_pe(pe_message):
    global pe_ltp, index_ltp, unsubscribe_done
    try:
        if pe_message['symbol'] == pe_symbol:
            if "ltp" in pe_message:
                pe_ltp = pe_message["ltp"]
                pe_ltp = float(pe_ltp)

        elif pe_message['symbol'] == index_symbol:
            if "ltp" in pe_message:
                index_ltp = pe_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [pe_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {pe_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessagePE): {e}")

def onerror_pe(message):
    print("PE Error:", message)


def onclose_pe(message):
    print("PE Connection closed:", message)


def onopen_pe():
    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [pe_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def pe_buy_sell_ltp():
    global buy_sell_checked, pe_symbol, pe_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching PE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if pe_symbol is not None and pe_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                pe_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_pe,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_pe,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_pe,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_pe             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                pe_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def place_order(symbol):
    try:
        market_order_data = {
            "symbol": symbol,
            "qty": int(quantity),
            "type": 2,  # Market Order
            "side": 1,
            "productType": "INTRADAY",
            "limitPrice": 0,
            "stopPrice": 0,
            "validity": "DAY",
            "disclosedQty": 0,
            "offlineOrder":False
        }

        market_order_entry = fyers.place_order(data=market_order_data)

        if "id" in market_order_entry:
            market_order_id = market_order_entry["id"]
            market_order_message = market_order_entry["message"]
            print(f"{market_order_message}")

    except Exception as e:
        print(f"Error placing orders: {str(e)}")

In [ ]:
def trail_order(symbol, stoploss):
    while True:
        try:
            stoploss = int(stoploss)
            pending_order = fyers.orderbook()

            matching_orders = [order for order in pending_order["orderBook"] if order["status"] == 6]

            modified_orders = 0

            for order in matching_orders:
                if order['symbol'] == symbol:
                    pending_order_id = order['id']
                    pending_order_side = order['side']
                    pending_order_side = int(pending_order_side)

                    if pending_order_side != 1:
                        data = {
                            "id": pending_order_id,
                            "type": 4,
                            "limitPrice": stoploss - 1,
                            "stopPrice": stoploss
                        }

                        modify = fyers.modify_order(data=data)
                        trail_message = modify["message"]
                        print(f"{trail_message}")

                        if trail_message == "Successfully modified order":
                            modified_orders += 1

            # Check if all matching orders are successfully modified
            if modified_orders == len(matching_orders):
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print("Error modifying order:" + str(e))

In [ ]:
def exit_active_order(symbol):
    while True:
        try:
            data = {
                "id":f"{symbol}-INTRADAY"
            }

            exit_response = fyers.exit_positions(data=data)

            if ["message"] in exit_response:
                print(exit_response["message"])
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print(f"Error exiting Order: {e}")

In [ ]:
def reset_flags():
    global active_order, buy_sell_checked

    active_order = False
    buy_sell_checked = False

In [ ]:
# Function to save profits and losses
def save_overall(overall_win, overall_loss, capital):
    trade_type = {
        "overall_win": overall_win,
        "overall_loss": overall_loss,
        "capital": capital
    }

    with open("trade_data.json", "w") as file:
        json.dump(trade_type, file)


# Function to load wins and losses
def load_overall():
    try:
        with open('trade_data.json') as file:
            return json.load(file)
    except FileNotFoundError:
        return None

In [ ]:
def handle_active_ce_order():
    def ce_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, ce_ltp, index_ltp, ce_strike, ce_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        ce_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if ce_ltp != 0 and index_ltp != 0:
                    ce_ltp_array.append(ce_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)


                    if index_ltp <= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(ce_symbol)

                        points = int(ce_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("CE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp >= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                            trailing_sl_inside = int(ce_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp - stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                    else:
                        if (index_ltp - prev_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(ce_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp - trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("CE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=ce_order_loop()).start()

In [ ]:
def handle_active_pe_order():
    def pe_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, pe_ltp, index_ltp, pe_strike, pe_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        pe_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if pe_ltp != 0 and index_ltp != 0:
                    pe_ltp_array.append(pe_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)

                    if index_ltp >= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(pe_symbol)

                        points = int(pe_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("PE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp_array}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp <= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                            trailing_sl_inside = int(pe_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp + stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                    else:
                        if (prev_ltp - index_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(pe_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp + trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("PE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=pe_order_loop()).start()

In [ ]:
def ce_entry():
    threading.Thread(target=ce_buy_sell_ltp).start()

    def ce_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if ce_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = ce_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp + target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp - trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(ce_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_ce_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=ce_entry_thread()).start()

In [ ]:
def pe_entry():
    threading.Thread(target=pe_buy_sell_ltp).start()

    def pe_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if pe_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = pe_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp - target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp + trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(pe_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_pe_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=pe_entry_thread()).start()

In [ ]:
def market_entry_exit_logic(rf_final_pred, actual_closing_price, predicted_price, final_df):
    global sl_hit_condition, unsubscribe_done, ce_ltp, pe_ltp, index_ltp, fixed_ltp, fixed_index_ltp, prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, ce_strike, pe_strike, ce_symbol, pe_symbol

    ce_ltp = 0
    pe_ltp  =0
    index_ltp = 0
    fixed_ltp = 0
    fixed_index_ltp = 0
    prev_ltp = 0
    target_inside = 0
    target_index_inside = 0
    trailing_sl_inside = 0
    trailing_index_inside = 0
    ce_strike = None
    pe_strike = None
    ce_symbol = None
    pe_symbol = None

    final_current_time = final_df.index[-1].time()

    if final_current_time >= dt_time(9, (15 + interval_minutes)) and final_current_time <= dt_time(15, 0):
        #CE entry
        if predicted_price > actual_closing_price and rf_final_pred == 2:
            if not active_order:
                pygame.mixer.music.play()

                sl_hit_condition = False
                unsubscribe_done = False

                ce_log_entry = f"CE Position: {final_df.index[-1]}, Actual Price: {actual_closing_price}, Predicted Price: {predicted_price}, RF Prediction: {rf_final_pred}\n"
                with open('market_entry_exit_log.txt', 'a') as log_file:
                    log_file.write(ce_log_entry)

                print("Making CE Position")
                ce_entry()

        #PE entry
        elif predicted_price < actual_closing_price and rf_final_pred == 1:
            if not active_order:
                pygame.mixer.music.play()

                sl_hit_condition = False
                unsubscribe_done = False

                pe_log_entry = f"PE Position: {final_df.index[-1]}, Actual Price: {actual_closing_price}, Predicted Price: {predicted_price}, RF Prediction: {rf_final_pred}\n"
                with open('market_entry_exit_log.txt', 'a') as log_file:
                    log_file.write(pe_log_entry)

                print("Making PE Position")
                pe_entry()

In [ ]:
def fetch_and_prepare_final_data():
    final_candle_data = fetch_candle_data(10)

    final_df = pd.DataFrame(final_candle_data['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])

    final_df = final_df.drop_duplicates(subset='datetime', keep='first')

    final_df = final_df[-(crop_fetched_candle_data + 1):]

    final_df = process_df_with_features(final_df)
    final_df = label_signals(final_df)

    final_df = final_df[[col for col in final_df.columns if col not in ['Entry Price', 'Exit Price']]]

    return final_df

In [ ]:
def get_trendline(df, point1, point2, kind='high'):
    x = [point1, point2]
    if kind == 'high':
        y = df['high'].values[x]
    else:
        y = df['low'].values[x]
    coeffs = np.polyfit(x, y, 1)
    trendline = np.polyval(coeffs, range(len(df)))
    return trendline

In [ ]:
#while True:
clear_output(wait=True)

num_candles = 100

final_df = fetch_and_prepare_final_data()
print(len(final_df))

final_df = final_df.iloc[:-1]

target = final_df['Target'].iloc[-1]

trailing_sl = final_df['Stop Loss'].iloc[-1]

# Identify most recent high and low points
recent_highs, recent_lows = find_local_extrema(final_df)

most_recent_high = recent_highs[-1] if len(recent_highs) > 1 else None
most_recent_low = recent_lows[-1] if len(recent_lows) > 1 else None

high_trendline = [np.nan] * len(final_df)
low_trendline = [np.nan] * len(final_df)

if most_recent_high is not None:
    previous_high = recent_highs[-2] if len(recent_highs) > 2 else most_recent_high
    high_trendline = get_trendline(final_df, previous_high, most_recent_high, kind='high')

if most_recent_low is not None:
    previous_low = recent_lows[-2] if len(recent_lows) > 2 else most_recent_low
    low_trendline = get_trendline(final_df, previous_low, most_recent_low, kind='low')

# Random Forest Classifier Prediction
rf_final_data = final_df[-1:][[col for col in final_df.columns if col != 'Signal']]

rf_final_pred = rf_model.predict(rf_final_data)
rf_final_pred = rf_final_pred[0]

#Scale the final df
scaled_final_df = final_df[-sequence_length:].copy()

for column in scaled_final_df.columns:
    scaled_final_df[column] = regression_scalers[column].transform(scaled_final_df[column].values.reshape(-1, 1))

# To include the most recent data for prediction:
latest_sequence = scaled_final_df.values
latest_sequence = latest_sequence.reshape(1, sequence_length, -1)

# Linear Regression Model Prediction
y_pred_lr_latest = lr_reg_model.predict(latest_sequence.reshape(latest_sequence.shape[0], -1))

# LSTM Model Prediction
y_pred_lstm_latest = lstm_reg_model.predict(latest_sequence)
y_pred_lstm_latest = np.squeeze(y_pred_lstm_latest)

# GRU Model Prediction
y_pred_gru_latest = gru_reg_model.predict(latest_sequence)
y_pred_gru_latest = np.squeeze(y_pred_gru_latest)

# Use the same weights as determined during training
y_pred_ensemble_latest = (weight_lr_reg * y_pred_lr_latest) + (weight_lstm_reg * y_pred_lstm_latest) + (weight_gru_reg * y_pred_gru_latest)

# Inverse transform prediction for Ensemble Model
y_pred_ensemble_latest_original = regression_scalers['close'].inverse_transform(y_pred_ensemble_latest.reshape(-1, 1)).flatten()
y_pred_ensemble_latest_original = round(y_pred_ensemble_latest_original[0], 2)

print(final_df.index[-1])
print(final_df['close'].iloc[-1])
print(y_pred_ensemble_latest_original)
print(rf_final_pred)

# Prepare candlestick data for mplfinance
actual_candles = final_df[-num_candles:].copy()

# Create a DataFrame for mplfinance
mpf_df = actual_candles[['open', 'high', 'low', 'close']]

# Create addplot elements for predicted prices and actual close prices
ap = [
    mpf.make_addplot(final_df['close'][-num_candles:], color='none', panel=0, secondary_y=False, label=f"Actual Price: {final_df['close'].iloc[-1]}\nPredicted Price:{y_pred_ensemble_latest_original}"),
    #mpf.make_addplot(y_pred_ensemble_final_plot, color=(0.95, 0.38, 0.25, 1), panel=0, secondary_y=False, label=f'Predicted Prices ({y_pred_ensemble_final_plot[-1]:.2f})')
]

# Add trendlines to the plot
if most_recent_high is not None:
    ap.append(mpf.make_addplot(high_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

if most_recent_low is not None:
    ap.append(mpf.make_addplot(low_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

fig, axlist = mpf.plot(mpf_df, type='candle', style='binancedark', volume=False, addplot=ap,
                        title=f'Chart', ylabel='Price',
                        figsize=(14, 7), returnfig=True)

for ax in axlist:
    ax.grid(False)

# Add the arrow for the future candle closing price
last_closing_price = final_df['close'].iloc[-1]
future_price = y_pred_ensemble_latest_original

if future_price > last_closing_price:
    arrow_text = '↑'
    arrow_color = 'green'
elif future_price < last_closing_price:
    arrow_text = '↓'
    arrow_color = 'red'
else:
    arrow_text = 'x'
    arrow_color = 'white'

axlist[0].annotate(
    arrow_text,
    (len(mpf_df), future_price),
    color=arrow_color,
    fontsize=20,
    fontweight='bold',
    ha='center'
)
axlist[0].legend()

plt.show()

#market_entry_exit_logic(rf_final_pred, final_df['close'].iloc[-1], y_pred_ensemble_latest_original, final_df)

#sleep_time = get_sleep_time(interval_minutes)
#time.sleep(sleep_time)